# COMP7035 Assignment 1: Programme Review

This notebook implements a complete data-cleaning workflow for `Programme Review from Messy Records`.

The implementation follows the consolidated strategy in `README.md`: keep identifiers as text, preserve audit trails, separate invalid data from conflicts, apply the policy cut-off before engagement reconciliation, and make recommendation boundary cases explicit.


In [ ]:
# =========================
# 0. Setup, constants, backup
# =========================

from pathlib import Path
from datetime import datetime
from decimal import Decimal
import json
import re
import shutil

import numpy as np
import pandas as pd

# Replace this with your own eight-digit student ID before final submission.
STUDENT_ID = "24301234"

DATA_DIR = Path.cwd()
DATA_FILES = [
    "students.json",
    "scores.csv",
    "engagement.log",
    "policy.json",
]

# Create a lightweight backup every time the notebook is run.
# Existing timestamped backups are kept, so rerunning the notebook does not destroy earlier copies.
run_stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
backup_dir = DATA_DIR / "backups" / f"notebook_run_{run_stamp}"
backup_dir.mkdir(parents=True, exist_ok=True)

for file_name in DATA_FILES + ["hw1_template.ipynb"]:
    source = DATA_DIR / file_name
    if source.exists():
        shutil.copy2(source, backup_dir / file_name)

print(f"Backup created: {backup_dir}")


## 1. Load Policy And Validate Global Rules

Before cleaning data, validate the policy file. The rest of the notebook assumes the score components and cut-off time come from this policy rather than being hard-coded in multiple places.


In [ ]:
# =========================
# 1. Load and validate policy
# =========================

with open(DATA_DIR / "policy.json", "r", encoding="utf-8") as f:
    policy = json.load(f)

required_components = ["midterm", "final", "project"]
weights_raw = policy.get("score_weights", {})

# Validate score weight fields.
if set(weights_raw) != set(required_components):
    raise ValueError(f"score_weights must contain exactly {required_components}; got {sorted(weights_raw)}")

# Decimal is used for auditable grade calculation and exact weight-sum checking.
weights = {component: Decimal(str(weights_raw[component])) for component in required_components}
if sum(weights.values(), Decimal("0")) != Decimal("1"):
    raise ValueError(f"Score weights must sum to 1; got {sum(weights.values(), Decimal('0'))}")

review_capacity = int(policy["review_capacity"])
if review_capacity <= 0:
    raise ValueError("review_capacity must be a positive integer")

as_of = pd.Timestamp(policy["as_of"])

print("Policy validated")
print("weights:", weights)
print("review_capacity:", review_capacity)
print("as_of:", as_of)


## 2. Clean Student Profiles

Rules applied here:

- keep `student_id` as text;
- trim IDs and names;
- normalize emails to lowercase;
- keep a profile-status flag for IDs that appear outside `students.json` later.


In [ ]:
# =========================
# 2. Clean students.json
# =========================

students_raw = pd.read_json(DATA_DIR / "students.json", dtype={"student_id": "string"})

students = (
    students_raw
    .assign(
        student_id_raw=lambda d: d["student_id"].astype("string"),
        student_id=lambda d: d["student_id"].astype("string").str.strip(),
        name=lambda d: d["name"].astype("string").str.strip(),
        email=lambda d: d["email"].astype("string").str.strip().str.lower(),
        profile_id_had_spaces=lambda d: d["student_id_raw"].ne(d["student_id"]),
    )
)

students["student_id_valid_8digit"] = students["student_id"].str.fullmatch(r"\d{8}", na=False)

if students["student_id"].duplicated().any():
    duplicated_ids = students.loc[students["student_id"].duplicated(keep=False), "student_id"].tolist()
    raise ValueError(f"Duplicate student profiles after trim: {duplicated_ids}")

profiles = students.set_index("student_id")[["name", "email", "profile_id_had_spaces", "student_id_valid_8digit"]]

print("Raw profile rows:", len(students_raw))
print("Unique profiles after trim:", profiles.index.nunique())
print("Profile IDs with surrounding spaces:", students.loc[students["profile_id_had_spaces"], "student_id"].tolist())


## 3. Clean Scores

The score file is intentionally messy. This section reads it as raw text first, so malformed rows, repeated headers, blank lines, and invalid values can all be audited with line numbers.


In [ ]:
# =========================
# 3. Clean scores.csv
# =========================

score_lines = pd.DataFrame({
    "line_no": np.arange(1, len((DATA_DIR / "scores.csv").read_text(encoding="utf-8").splitlines()) + 1),
    "raw_line": (DATA_DIR / "scores.csv").read_text(encoding="utf-8").splitlines(),
})

# Split without forcing a strict CSV schema, because some rows have missing or extra fields.
score_parts = score_lines["raw_line"].str.split(",", expand=True)
score_parts = score_parts.reindex(columns=range(max(5, score_parts.shape[1])))
score_parts = score_parts.apply(lambda col: col.astype("string").str.strip())

scores_parsed = score_lines.assign(
    n_fields=score_parts.notna().sum(axis=1),
    student_id=score_parts[0],
    midterm_raw=score_parts[1],
    final_raw=score_parts[2],
    project_raw=score_parts[3],
)

scores_parsed["is_blank"] = scores_parsed["raw_line"].str.strip().eq("")
scores_parsed["is_header"] = (
    scores_parsed["student_id"].eq("student_id")
    & scores_parsed["midterm_raw"].eq("midterm")
    & scores_parsed["final_raw"].eq("final")
    & scores_parsed["project_raw"].eq("project")
)
scores_parsed["is_data"] = ~(scores_parsed["is_blank"] | scores_parsed["is_header"])
scores_parsed["student_id_valid_8digit"] = scores_parsed["student_id"].str.fullmatch(r"\d{8}", na=False)
scores_parsed["malformed_score_row"] = scores_parsed["is_data"] & scores_parsed["n_fields"].ne(4)

score_data = scores_parsed.loc[scores_parsed["is_data"]].copy()

# Convert the wide score columns into a long table so validation is vectorized by component.
score_long = score_data.melt(
    id_vars=["line_no", "raw_line", "student_id", "student_id_valid_8digit", "malformed_score_row"],
    value_vars=["midterm_raw", "final_raw", "project_raw"],
    var_name="component_raw",
    value_name="raw_value",
)
score_long["component"] = score_long["component_raw"].str.replace("_raw", "", regex=False)
score_long["raw_value"] = score_long["raw_value"].astype("string").str.strip()
score_long["score_numeric"] = pd.to_numeric(score_long["raw_value"], errors="coerce")

score_long["invalid_score_reason"] = np.select(
    [
        score_long["raw_value"].isna() | score_long["raw_value"].eq(""),
        score_long["score_numeric"].isna(),
        (score_long["score_numeric"].lt(0) | score_long["score_numeric"].gt(100)).fillna(False),
    ],
    ["missing_score", "non_numeric_score", "out_of_range_score"],
    default="",
)

valid_scores_long = score_long.loc[
    score_long["student_id_valid_8digit"]
    & score_long["invalid_score_reason"].eq("")
].copy()

# Baseline rule: if multiple valid values conflict, use the lower value while keeping all candidates for audit.
score_value_sets = (
    valid_scores_long
    .groupby(["student_id", "component"], as_index=False)
    .agg(
        accepted_score=("score_numeric", "min"),
        high_score_candidate=("score_numeric", "max"),
        candidate_values=("score_numeric", lambda s: ";".join(map(lambda x: f"{x:g}", sorted(set(s))))),
        candidate_line_nos=("line_no", lambda s: ";".join(map(str, sorted(set(s))))),
        valid_value_count=("score_numeric", "nunique"),
    )
)
score_value_sets["score_conflict"] = score_value_sets["valid_value_count"].gt(1)

scores_wide = (
    score_value_sets
    .pivot(index="student_id", columns="component", values="accepted_score")
    .reindex(columns=required_components)
    .reset_index()
)

score_conflicts = score_value_sets.loc[score_value_sets["score_conflict"]].copy()
score_missing_or_invalid = score_long.loc[score_long["invalid_score_reason"].ne("")].copy()
score_malformed_rows = scores_parsed.loc[scores_parsed["malformed_score_row"]].copy()

print("Score data rows:", len(score_data))
print("Repeated header rows:", int(scores_parsed["is_header"].sum()))
print("Blank rows:", int(scores_parsed["is_blank"].sum()))
print("Malformed score rows:", len(score_malformed_rows))
print("Invalid score values:", len(score_missing_or_invalid))
print("Students with score conflicts:", sorted(score_conflicts["student_id"].unique()))


## 4. Clean Engagement Records

The engagement log is parsed as raw text to preserve malformed records. The policy cut-off is applied before conflict detection, because post-cut-off records should not create baseline conflicts.


In [ ]:
# =========================
# 4. Clean engagement.log
# =========================

eng_lines_raw = (DATA_DIR / "engagement.log").read_text(encoding="utf-8").splitlines()
eng_lines = pd.DataFrame({
    "line_no": np.arange(1, len(eng_lines_raw) + 1),
    "raw_line": eng_lines_raw,
})

eng_lines["stripped"] = eng_lines["raw_line"].str.strip()
eng_candidate = eng_lines.loc[
    eng_lines["stripped"].ne("") & ~eng_lines["stripped"].str.startswith("#")
].copy()

eng_parts = eng_candidate["raw_line"].str.split("|", expand=True)
eng_parts = eng_parts.reindex(columns=range(max(4, eng_parts.shape[1])))
eng_parts = eng_parts.apply(lambda col: col.astype("string").str.strip())

eng = eng_candidate.assign(
    n_fields=eng_parts.notna().sum(axis=1),
    recorded_at_raw=eng_parts[0],
    student_id=eng_parts[1],
    week=eng_parts[2],
    sessions_raw=eng_parts[3],
)

eng["recorded_at"] = pd.to_datetime(eng["recorded_at_raw"], errors="coerce")
eng["sessions_numeric"] = pd.to_numeric(eng["sessions_raw"], errors="coerce")
eng["sessions_is_integer"] = eng["sessions_raw"].astype("string").str.fullmatch(r"[-+]?\d+", na=False)
eng["student_id_valid_8digit"] = eng["student_id"].str.fullmatch(r"\d{8}", na=False)
eng["week_valid"] = eng["week"].isin(["w1", "w2", "w3", "w4"])
eng["sessions_valid"] = eng["sessions_is_integer"] & eng["sessions_numeric"].between(0, 2, inclusive="both")

eng["invalid_engagement_reason"] = np.select(
    [
        eng["n_fields"].ne(4),
        eng["recorded_at"].isna(),
        ~eng["student_id_valid_8digit"],
        ~eng["week_valid"],
        ~eng["sessions_valid"],
    ],
    ["wrong_field_count", "invalid_timestamp", "invalid_student_id", "invalid_week", "invalid_sessions"],
    default="",
)

eng_valid = eng.loc[eng["invalid_engagement_reason"].eq("")].copy()
eng_valid["sessions"] = eng_valid["sessions_numeric"].astype("Int64")
eng_valid["post_cutoff"] = eng_valid["recorded_at"].gt(as_of)

eng_baseline = eng_valid.loc[~eng_valid["post_cutoff"]].copy()
eng_post_cutoff = eng_valid.loc[eng_valid["post_cutoff"]].copy()
eng_invalid = eng.loc[eng["invalid_engagement_reason"].ne("")].copy()

engagement_value_sets = (
    eng_baseline
    .groupby(["student_id", "week"], as_index=False)
    .agg(
        accepted_sessions=("sessions", "max"),
        candidate_sessions=("sessions", lambda s: ";".join(map(str, sorted(set(s.dropna().astype(int)))))),
        candidate_line_nos=("line_no", lambda s: ";".join(map(str, sorted(set(s))))),
        session_value_count=("sessions", "nunique"),
    )
)
engagement_value_sets["engagement_conflict"] = engagement_value_sets["session_value_count"].gt(1)

same_timestamp_conflicts = (
    eng_baseline
    .groupby(["student_id", "week", "recorded_at"], as_index=False)
    .agg(session_value_count=("sessions", "nunique"), line_nos=("line_no", lambda s: ";".join(map(str, sorted(set(s))))))
    .query("session_value_count > 1")
)

engagement_wide = (
    engagement_value_sets
    .pivot(index="student_id", columns="week", values="accepted_sessions")
    .reindex(columns=["w1", "w2", "w3", "w4"])
    .rename(columns={week: f"{week}_sessions" for week in ["w1", "w2", "w3", "w4"]})
    .reset_index()
)

week_cols = ["w1_sessions", "w2_sessions", "w3_sessions", "w4_sessions"]
engagement_wide["weeks_recorded"] = engagement_wide[week_cols].notna().sum(axis=1)
engagement_wide["weeks_unknown"] = 4 - engagement_wide["weeks_recorded"]
engagement_wide["sessions_total_known"] = engagement_wide[week_cols].sum(axis=1, min_count=1)

print("Valid engagement IDs including post-cutoff:", eng_valid["student_id"].nunique())
print("Invalid engagement rows:", len(eng_invalid))
print("Post-cutoff engagement rows:", len(eng_post_cutoff))
print("Baseline engagement conflicts:", int(engagement_value_sets["engagement_conflict"].sum()))
print("Same-timestamp conflicts:", len(same_timestamp_conflicts))


## 5. Build Audit Log

The audit log records data that was ignored, corrected, resolved by rule, or left for human review. This makes the final recommendation reproducible and reviewable.


In [ ]:
# =========================
# 5. Build audit log
# =========================

audit_frames = []

# Score-level audit entries.
audit_frames.append(
    scores_parsed.loc[scores_parsed["is_header"], ["line_no", "raw_line"]]
    .assign(source_file="scores.csv", issue="repeated_score_header", student_id=pd.NA, field=pd.NA, raw_value=pd.NA)
)
audit_frames.append(
    scores_parsed.loc[scores_parsed["is_blank"], ["line_no", "raw_line"]]
    .assign(source_file="scores.csv", issue="blank_score_line", student_id=pd.NA, field=pd.NA, raw_value=pd.NA)
)
audit_frames.append(
    score_malformed_rows[["line_no", "raw_line", "student_id"]]
    .assign(source_file="scores.csv", issue="malformed_score_row", field=pd.NA, raw_value=pd.NA)
)
audit_frames.append(
    score_missing_or_invalid[["line_no", "raw_line", "student_id", "component", "raw_value", "invalid_score_reason"]]
    .rename(columns={"component": "field", "invalid_score_reason": "issue"})
    .assign(source_file="scores.csv")
)
audit_frames.append(
    score_conflicts.assign(
        line_no=lambda d: d["candidate_line_nos"],
        raw_line=pd.NA,
        field=lambda d: d["component"],
        raw_value=lambda d: d["candidate_values"],
        issue="score_conflict",
        source_file="scores.csv",
    )[["source_file", "line_no", "raw_line", "student_id", "field", "raw_value", "issue"]]
)

# Engagement-level audit entries.
audit_frames.append(
    eng_invalid[["line_no", "raw_line", "student_id", "week", "sessions_raw", "invalid_engagement_reason"]]
    .rename(columns={"week": "field", "sessions_raw": "raw_value", "invalid_engagement_reason": "issue"})
    .assign(source_file="engagement.log")
)
audit_frames.append(
    eng_post_cutoff[["line_no", "raw_line", "student_id", "week", "sessions"]]
    .rename(columns={"week": "field", "sessions": "raw_value"})
    .assign(source_file="engagement.log", issue="post_cutoff_engagement")
)
audit_frames.append(
    engagement_value_sets.loc[engagement_value_sets["engagement_conflict"]]
    .assign(
        source_file="engagement.log",
        line_no=lambda d: d["candidate_line_nos"],
        raw_line=pd.NA,
        field=lambda d: d["week"],
        raw_value=lambda d: d["candidate_sessions"],
        issue="engagement_conflict",
    )[["source_file", "line_no", "raw_line", "student_id", "field", "raw_value", "issue"]]
)
audit_frames.append(
    same_timestamp_conflicts.assign(
        source_file="engagement.log",
        line_no=lambda d: d["line_nos"],
        raw_line=pd.NA,
        field=lambda d: d["week"],
        raw_value=lambda d: d["recorded_at"].astype("string"),
        issue="same_timestamp_conflict",
    )[["source_file", "line_no", "raw_line", "student_id", "field", "raw_value", "issue"]]
)

# Combine all audit entries into one tidy table.
audit_log = pd.concat(audit_frames, ignore_index=True, sort=False)
audit_log = audit_log[["source_file", "line_no", "student_id", "field", "raw_value", "issue", "raw_line"]]

print("Audit entries:", len(audit_log))
audit_log["issue"].value_counts(dropna=False)


## 6. Merge Clean Data And Generate Recommendations

The final table uses the union of valid eight-digit IDs from profiles, scores, and engagement. Missing profile, score quality, engagement quality, and boundary-risk flags are all carried forward into `review_reasons`.


In [ ]:
# =========================
# 6. Merge clean data and generate recommendations
# =========================

profile_ids = pd.Index(profiles.index.astype("string"))
score_ids = pd.Index(score_data.loc[score_data["student_id_valid_8digit"], "student_id"].dropna().unique().astype("string"))
engagement_ids = pd.Index(eng_valid["student_id"].dropna().unique().astype("string"))

all_valid_ids = profile_ids.union(score_ids).union(engagement_ids).sort_values()
clean = pd.DataFrame({"student_id": all_valid_ids})

clean = (
    clean
    .merge(profiles.reset_index(), on="student_id", how="left")
    .merge(scores_wide, on="student_id", how="left")
    .merge(engagement_wide, on="student_id", how="left")
)

score_component_cols = required_components
clean["has_profile"] = clean["name"].notna()
clean["missing_profile"] = ~clean["has_profile"]
clean["has_any_score_record"] = clean["student_id"].isin(score_ids)
clean["has_any_engagement_record"] = clean["student_id"].isin(engagement_ids)
clean["profile_only_no_records"] = clean["has_profile"] & ~clean["has_any_score_record"] & ~clean["has_any_engagement_record"]
clean["score_complete"] = clean[score_component_cols].notna().all(axis=1)
clean["score_missing"] = clean["has_any_score_record"] & ~clean["score_complete"]

score_conflict_ids = pd.Index(score_conflicts["student_id"].unique())
engagement_conflict_ids = pd.Index(engagement_value_sets.loc[engagement_value_sets["engagement_conflict"], "student_id"].unique())
same_ts_conflict_ids = pd.Index(same_timestamp_conflicts["student_id"].unique())
post_cutoff_ids = pd.Index(eng_post_cutoff["student_id"].unique())

clean["score_conflict"] = clean["student_id"].isin(score_conflict_ids)
clean["engagement_conflict"] = clean["student_id"].isin(engagement_conflict_ids)
clean["same_timestamp_conflict"] = clean["student_id"].isin(same_ts_conflict_ids)
clean["has_post_cutoff_engagement"] = clean["student_id"].isin(post_cutoff_ids)
clean["possible_unmerged_short_id_engagement"] = clean["student_id"].eq("04770215")

# Decimal calculation keeps the policy calculation auditable.
def weighted_total_decimal(row):
    if not bool(row["score_complete"]):
        return pd.NA
    total = sum(Decimal(str(row[component])) * weights[component] for component in required_components)
    return total.quantize(Decimal("0.01"))

clean["weighted_total_decimal"] = clean.apply(weighted_total_decimal, axis=1)
clean["weighted_total"] = pd.to_numeric(clean["weighted_total_decimal"].astype("string"), errors="coerce")

# Baseline recommendation: only complete-score students with verified profiles are eligible.
eligible = clean.loc[clean["score_complete"] & clean["has_profile"]].copy()
eligible = eligible.sort_values(["weighted_total", "student_id"], ascending=[False, True])
eligible["review_rank"] = np.arange(1, len(eligible) + 1)
eligible["recommended_for_review"] = eligible["review_rank"].le(review_capacity)

clean = clean.merge(
    eligible[["student_id", "review_rank", "recommended_for_review"]],
    on="student_id",
    how="left",
)
clean["recommended_for_review"] = clean["recommended_for_review"].fillna(False)

# Boundary definition: within 1 point of rank 15 or rank 16, or affected by score conflicts.
rank_15_score = eligible.loc[eligible["review_rank"].eq(review_capacity), "weighted_total"].squeeze() if len(eligible) >= review_capacity else np.nan
rank_16_score = eligible.loc[eligible["review_rank"].eq(review_capacity + 1), "weighted_total"].squeeze() if len(eligible) >= review_capacity + 1 else np.nan
boundary_scores = pd.Series([rank_15_score, rank_16_score], dtype="float64").dropna()

if boundary_scores.empty:
    clean["near_recommendation_boundary"] = False
else:
    # Vectorized boundary check: compare every student's score against rank 15 and rank 16 scores.
    score_matrix = clean["weighted_total"].to_numpy(dtype="float64")[:, None]
    boundary_matrix = boundary_scores.to_numpy(dtype="float64")[None, :]
    clean["near_recommendation_boundary"] = np.any(np.abs(score_matrix - boundary_matrix) <= 1, axis=1)

clean["tie_at_cutoff"] = False
if not pd.isna(rank_15_score) and not pd.isna(rank_16_score) and rank_15_score == rank_16_score:
    clean["tie_at_cutoff"] = clean["weighted_total"].eq(rank_15_score)

# Human-readable review reasons. This is intentionally explicit rather than clever.
def build_review_reasons(row):
    reasons = []
    if row["missing_profile"]:
        reasons.append("missing_profile")
    if row["profile_only_no_records"]:
        reasons.append("profile_only_no_records")
    if row["score_missing"]:
        reasons.append("score_missing_or_incomplete")
    if row["score_conflict"]:
        reasons.append("score_conflict")
    if row["engagement_conflict"]:
        reasons.append("engagement_conflict")
    if row["same_timestamp_conflict"]:
        reasons.append("same_timestamp_conflict")
    if row["has_post_cutoff_engagement"]:
        reasons.append("post_cutoff_engagement")
    if row["possible_unmerged_short_id_engagement"]:
        reasons.append("possible_unmerged_short_id_engagement")
    if row["near_recommendation_boundary"]:
        reasons.append("near_recommendation_boundary")
    if row["tie_at_cutoff"]:
        reasons.append("tie_at_cutoff")
    return ";".join(reasons)

clean["review_reasons"] = clean.apply(build_review_reasons, axis=1)
clean["needs_human_review"] = clean["review_reasons"].ne("")

# Order columns for readability.
final_columns = [
    "student_id", "name", "email",
    "midterm", "final", "project", "weighted_total",
    "w1_sessions", "w2_sessions", "w3_sessions", "w4_sessions",
    "weeks_recorded", "weeks_unknown", "sessions_total_known",
    "recommended_for_review", "review_rank", "tie_at_cutoff",
    "needs_human_review", "review_reasons",
]
clean_final = clean.reindex(columns=final_columns).sort_values("student_id").reset_index(drop=True)

print("Final student rows:", len(clean_final))
print("Recommended students:", int(clean_final["recommended_for_review"].sum()))
print("Needs human review:", int(clean_final["needs_human_review"].sum()))
clean_final.loc[clean_final["recommended_for_review"]].sort_values("review_rank")


## 7. Meaningful Question

Question: do students with more recorded engagement sessions also tend to have higher weighted scores?

This is useful because engagement is not used as a score bonus in the recommendation rule, but it can still provide context for programme review.


In [ ]:
# =========================
# 7. Meaningful question: engagement vs score
# =========================

analysis_df = clean.loc[clean["score_complete"] & clean["sessions_total_known"].notna()].copy()

engagement_summary = (
    analysis_df
    .assign(engagement_band=pd.cut(
        analysis_df["sessions_total_known"],
        bins=[-0.1, 2, 5, 8],
        labels=["low", "medium", "high"],
    ))
    .groupby("engagement_band", observed=False)
    .agg(
        students=("student_id", "count"),
        avg_weighted_total=("weighted_total", "mean"),
        median_weighted_total=("weighted_total", "median"),
        avg_weeks_recorded=("weeks_recorded", "mean"),
    )
    .round(2)
)

correlation = analysis_df[["sessions_total_known", "weighted_total"]].corr(method="spearman").iloc[0, 1]

print("Spearman correlation between known sessions and weighted total:", round(correlation, 3))
engagement_summary


## 8. Additional Analytical Experiment

Experiment: change the score-conflict rule from “take the lower valid value” to “take the higher valid value”, then compare whether the recommendation list changes.


In [ ]:
# =========================
# 8. Additional experiment: lower-value vs higher-value score conflicts
# =========================

scores_high_wide = (
    score_value_sets
    .pivot(index="student_id", columns="component", values="high_score_candidate")
    .reindex(columns=required_components)
    .reset_index()
)

experiment = (
    clean[["student_id", "name", "email", "has_profile"]]
    .merge(scores_high_wide, on="student_id", how="left")
)
experiment["score_complete"] = experiment[required_components].notna().all(axis=1)

def weighted_total_decimal_high(row):
    if not bool(row["score_complete"]):
        return pd.NA
    total = sum(Decimal(str(row[component])) * weights[component] for component in required_components)
    return total.quantize(Decimal("0.01"))

experiment["weighted_total_high_conflict_rule"] = pd.to_numeric(
    experiment.apply(weighted_total_decimal_high, axis=1).astype("string"),
    errors="coerce",
)

eligible_high = experiment.loc[experiment["score_complete"] & experiment["has_profile"]].copy()
eligible_high = eligible_high.sort_values(["weighted_total_high_conflict_rule", "student_id"], ascending=[False, True])
eligible_high["review_rank_high_rule"] = np.arange(1, len(eligible_high) + 1)
eligible_high["recommended_high_rule"] = eligible_high["review_rank_high_rule"].le(review_capacity)

baseline_recommended = set(clean.loc[clean["recommended_for_review"], "student_id"])
high_rule_recommended = set(eligible_high.loc[eligible_high["recommended_high_rule"], "student_id"])

recommendation_comparison = pd.DataFrame({
    "student_id": sorted(baseline_recommended.union(high_rule_recommended)),
})
recommendation_comparison["baseline_recommended"] = recommendation_comparison["student_id"].isin(baseline_recommended)
recommendation_comparison["high_rule_recommended"] = recommendation_comparison["student_id"].isin(high_rule_recommended)
recommendation_comparison = recommendation_comparison.merge(
    clean[["student_id", "weighted_total", "review_rank", "score_conflict", "review_reasons"]],
    on="student_id",
    how="left",
).merge(
    eligible_high[["student_id", "weighted_total_high_conflict_rule", "review_rank_high_rule"]],
    on="student_id",
    how="left",
)
recommendation_comparison["changed"] = recommendation_comparison["baseline_recommended"].ne(recommendation_comparison["high_rule_recommended"])

print("Recommendation changed under high-conflict rule:", bool(recommendation_comparison["changed"].any()))
recommendation_comparison.sort_values(["changed", "baseline_recommended", "student_id"], ascending=[False, False, True])


## 9. Deliverables And Final Check

This section writes the clean student-level CSV and a separate audit CSV. The clean CSV is the main deliverable; the audit CSV supports traceability.


In [ ]:
# =========================
# 9. Write deliverables
# =========================

clean_csv_path = DATA_DIR / f"hw1_{STUDENT_ID}_clean.csv"
audit_csv_path = DATA_DIR / f"hw1_{STUDENT_ID}_audit.csv"

# Use utf-8-sig so Excel can open the file cleanly while preserving text IDs.
clean_final.to_csv(clean_csv_path, index=False, encoding="utf-8-sig")
audit_log.to_csv(audit_csv_path, index=False, encoding="utf-8-sig")

print("Clean CSV written:", clean_csv_path)
print("Audit CSV written:", audit_csv_path)
print("Clean shape:", clean_final.shape)
print("Audit shape:", audit_log.shape)

# Display a compact final quality summary.
quality_summary = pd.Series({
    "final_student_rows": len(clean_final),
    "missing_profile_rows": int(clean["missing_profile"].sum()),
    "profile_only_no_records_rows": int(clean["profile_only_no_records"].sum()),
    "score_conflict_students": int(clean["score_conflict"].sum()),
    "engagement_conflict_students": int(clean["engagement_conflict"].sum()),
    "same_timestamp_conflict_students": int(clean["same_timestamp_conflict"].sum()),
    "post_cutoff_engagement_students": int(clean["has_post_cutoff_engagement"].sum()),
    "recommended_for_review": int(clean["recommended_for_review"].sum()),
    "tie_at_cutoff_students": int(clean["tie_at_cutoff"].sum()),
})
quality_summary
